## Сканирование коэффициента заразности β

### Описание

Данный скрипт исследует, как изменение базовой заразности (β) влияет на
эпидемические показатели: пик заболеваемости, долю переболевших и число умерших.
Выполняется параметрическое сканирование с несколькими повторными прогонами
для учёта стохастичности.

In [ ]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots, CSV, Random, Statistics

include(srcdir("sir_model.jl"))

### Функция для запуска одного эксперимента и возврата метрик

Функция принимает словарь параметров `p` и возвращает:
- `peak`: пиковая доля инфицированных
- `final_inf`: конечная доля инфицированных
- `final_rec`: конечная доля выздоровевших
- `deaths`: общее число умерших

In [ ]:
function run_experiment(p)
    beta = p[:beta]
    β_und = fill(beta, 3)
    β_det = fill(beta/10, 3)

    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = β_und,
        β_det = β_det,
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
    )

    infected_fraction(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    peak_infected = 0.0

    for step in 1:p[:n_steps]
        Agents.step!(model, 1)
        frac = infected_fraction(model)
        if frac > peak_infected
            peak_infected = frac
        end
    end

    final_infected = infected_fraction(model)
    final_recovered = count(a.status == :R for a in allagents(model)) / nagents(model)
    total_deaths = sum(p[:Ns]) - nagents(model)

    return (
        peak = peak_infected,
        final_inf = final_infected,
        final_rec = final_recovered,
        deaths = total_deaths,
    )
end

### Параметры сканирования

- Диапазон $\beta$: от 0.1 до 1.0 с шагом 0.1
- Количество повторных прогонов для каждого $\beta$: 3 (разные seed)

In [ ]:
beta_range = 0.1:0.1:1.0
seeds = [42, 43, 44]

### Формирование списка параметров

In [ ]:
params_list = []
for b in beta_range
    for s in seeds
        push!(params_list, Dict(
            :beta => b,
            :Ns => [1000, 1000, 1000],
            :infection_period => 14,
            :detection_time => 7,
            :death_rate => 0.02,
            :reinfection_probability => 0.1,
            :Is => [0, 0, 1],
            :seed => s,
            :n_steps => 100,
        ))
    end
end

### Запуск экспериментов

In [ ]:
results = []
for params in params_list
    data = run_experiment(params)
    push!(results, merge(params, Dict(pairs(data))))
    println("Завершён эксперимент с beta = $(params[:beta]), seed = $(params[:seed])")
end

### Сохранение результатов

In [ ]:
df = DataFrame(results)
CSV.write(datadir("beta_scan_all.csv"), df)

### Усреднение по повторным прогонам

In [ ]:
grouped = combine(groupby(df, [:beta]),
    :peak => mean => :mean_peak,
    :final_inf => mean => :mean_final_inf,
    :deaths => mean => :mean_deaths,
)

### Визуализация

In [ ]:
plot(grouped.beta, grouped.mean_peak,
     label = "Пик эпидемии",
     xlabel = "Коэффициент заразности β",
     ylabel = "Доля инфицированных",
     marker = :circle,
     linewidth = 2)
plot!(grouped.beta, grouped.mean_final_inf,
      label = "Конечная доля инфицированных",
      marker = :square)
plot!(grouped.beta, grouped.mean_deaths ./ 3000,
      label = "Доля умерших",
      marker = :diamond)
savefig(plotsdir("beta_scan.png"))

println("Результаты сохранены в data/beta_scan_all.csv и plots/beta_scan.png")

### Выводы

1. Сканирование коэффициента заразности $\beta$ позволяет выявить пороговое
   значение, при котором возникает эпидемия. Теоретический порог соответствует
   $R_0 = 1$, т.е. $\beta = \gamma$.

2. С ростом $\beta$ наблюдается нелинейное увеличение пиковой доли
   инфицированных: чем выше заразность, тем быстрее распространяется инфекция
   и тем больше людей одновременно болеют.

3. Конечная доля инфицированных (переболевших) также возрастает с увеличением
   $\beta$, стремясь к насыщению, что связано с формированием популяционного
   иммунитета.

4. Доля умерших коррелирует с пиком заболеваемости: более интенсивная эпидемия
   приводит к большему числу летальных исходов.

5. Наличие повторных прогонов с разными seed позволяет оценить стохастическую
   изменчивость результатов, что особенно важно при малых значениях $\beta$,
   где случайные флуктуации могут существенно влиять на исход.

6. Полученные данные сохраняются в CSV-файл для дальнейшего анализа,
   а график наглядно демонстрирует зависимость ключевых показателей
   от заразности инфекции.